In [3]:
import numpy as np
import pandas as pd
import torch

from sklearn.metrics import (
    accuracy_score,
    f1_score,
    balanced_accuracy_score,
    matthews_corrcoef
)

In [4]:
train_df = pd.read_csv("../data/processed_data/train.csv")
dev_df = pd.read_csv("../data/processed_data/dev.csv")

label2id = {
    "not depression": 0,
    "moderate": 1,
    "severe": 2
}

id2label = {v: k for k, v in label2id.items()}

train_df = train_df.dropna(subset=["Text_data", "Label"]).copy()
dev_df = dev_df.dropna(subset=["Text_data", "Label"]).copy()

train_df["Text_data"] = train_df["Text_data"].astype(str)
dev_df["Text_data"] = dev_df["Text_data"].astype(str)

train_df["labels"] = train_df["Label"].map(label2id)
dev_df["labels"] = dev_df["Label"].map(label2id)

assert train_df["labels"].notna().all()
assert dev_df["labels"].notna().all()

train_df["labels"] = train_df["labels"].astype(int)
dev_df["labels"] = dev_df["labels"].astype(int)

print("Training samples:", len(train_df))
print("Development samples:", len(dev_df))
print("\nTraining label distribution:")
print(train_df["Label"].value_counts())

print("\nDevelopment label distribution:")
print(dev_df["Label"].value_counts())

Training samples: 5488
Development samples: 1373

Training label distribution:
Label
moderate          2807
not depression    2116
severe             565
Name: count, dtype: int64

Development label distribution:
Label
moderate          703
not depression    529
severe            141
Name: count, dtype: int64


In [5]:
from transformers import AutoTokenizer

MODEL_NAME = "microsoft/deberta-v3-base"

tokenizer = AutoTokenizer.from_pretrained(
    MODEL_NAME,
    use_fast=False
)

print("Tokenizer loaded successfully!")
print("Vocabulary size:", tokenizer.vocab_size)

sample_text = train_df["Text_data"].iloc[0]
sample_tokens = tokenizer(
    sample_text,
    truncation=True,
    max_length=256
)

print("Sample token count:", len(sample_tokens["input_ids"]))

Tokenizer loaded successfully!
Vocabulary size: 128000
Sample token count: 256


/Users/kavya/SVNIT/sem5/ml_proj/.venv/lib/python3.12/site-packages/transformers/tokenization_utils_base.py:1601: FutureWarning: `clean_up_tokenization_spaces` was not set. It will be set to `True` by default. This behavior will be depracted in transformers v4.45, and will be then set to `False` by default. For more details check this issue: https://github.com/huggingface/transformers/issues/31884
  warnings.warn(


In [6]:
from datasets import Dataset

# Convert pandas DataFrames into Hugging Face Datasets
train_dataset = Dataset.from_pandas(
    train_df[["Text_data", "Label"]].reset_index(drop=True)
)

dev_dataset = Dataset.from_pandas(
    dev_df[["Text_data", "Label"]].reset_index(drop=True)
)

# Convert string labels into numerical labels
def encode_labels(example):
    example["labels"] = label2id[example["Label"]]
    return example

train_dataset = train_dataset.map(encode_labels)
dev_dataset = dev_dataset.map(encode_labels)

# Tokenize the text
def tokenize_function(examples):
    return tokenizer(
        examples["Text_data"],
        truncation=True,
        padding="max_length",
        max_length=MAX_LENGTH
    )

train_dataset = train_dataset.map(
    tokenize_function,
    batched=True,
    remove_columns=["Text_data", "Label"]
)

dev_dataset = dev_dataset.map(
    tokenize_function,
    batched=True,
    remove_columns=["Text_data", "Label"]
)

print("Tokenization complete!")
print("Training samples:", len(train_dataset))
print("Validation samples:", len(dev_dataset))
print("Features:", train_dataset.column_names)

ImportError: cannot import name 'BucketNotFoundError' from 'huggingface_hub.errors' (/Users/kavya/SVNIT/sem5/ml_proj/.venv/lib/python3.12/site-packages/huggingface_hub/errors.py)

In [7]:
import transformers
import huggingface_hub
import datasets
import tokenizers

print("Transformers:", transformers.version)
print("Hugging Face Hub:", huggingface_hub.version)
print("Datasets:", datasets.version)
print(“Tokenizers:”, tokenizers.version)

SyntaxError: invalid character '“' (U+201C) (454654879.py, line 6)